# Semantic Gaussians: CPU preparation and GPU fusion on Colab

Run the **CPU preparation** section on a free CPU runtime first. After all inputs are verified in Google Drive, disconnect that runtime, connect a **T4 GPU** runtime, and continue from the GPU section. Dataset, checkpoint, OpenSeg weights, and final features persist in Drive.

In [1]:
print('hi')

hi


In [ ]:
# CPU 1. Import preparation tools. This section does not require a GPU.
import shutil
import subprocess
import sys
import urllib.request

print("CPU preparation can run without consuming GPU compute units.")

In [ ]:
# CPU 2. Mount Drive and define persistent paths.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
REPO = Path("/content/semantic-gaussians")
CONDA = Path("/content/miniforge3/bin/conda")
DRIVE_ROOT = Path("/content/drive/MyDrive/semantic-gaussians")
DRIVE_DATASET = DRIVE_ROOT / "data/bonsai"
DRIVE_CHECKPOINT = DRIVE_ROOT / "bonsai/checkpoint"
DRIVE_OPENSEG = DRIVE_ROOT / "weights/openseg_exported_clip"
DRIVE_OUTPUT = DRIVE_ROOT / "outputs/bonsai/fused"
for path in (DRIVE_DATASET, DRIVE_CHECKPOINT, DRIVE_OPENSEG, DRIVE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)
print("Persistent dataset:", DRIVE_DATASET)
print("Persistent checkpoint:", DRIVE_CHECKPOINT)
print("Persistent OpenSeg model:", DRIVE_OPENSEG)
print("Persistent output:", DRIVE_OUTPUT)

In [ ]:
# CPU 3. Download the Bonsai images and COLMAP files directly to Drive.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
    check=True,
)
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="rishitdagli/nerf-gs-datasets",
    repo_type="dataset",
    allow_patterns=["bonsai/images_4/*", "bonsai/sparse/0/*", "bonsai/nb-info.json"],
    local_dir=DRIVE_ROOT / "data",
)
print("Dataset download finished.")

In [ ]:
# CPU 4. Download the pretrained Gaussian checkpoint to Drive once.
checkpoint_ply = DRIVE_CHECKPOINT / "point_cloud/iteration_30000/point_cloud.ply"
if not checkpoint_ply.is_file():
    archive = Path("/content/bonsai-checkpoint.zip")
    subprocess.run(
        ["wget", "-c",
         "https://data.ciirc.cvut.cz/public/projects/2023NerfBaselines/data/gaussian-splatting/mipnerf360/bonsai.zip",
         "-O", str(archive)], check=True,
    )
    (DRIVE_ROOT / "bonsai").mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["unzip", "-q", "-o", str(archive), "-d", str(DRIVE_ROOT / "bonsai")],
        check=True,
    )
else:
    print("Checkpoint already cached; skipping download.")
print("Checkpoint ready:", checkpoint_ply)

In [ ]:
# CPU 5. Download OpenSeg directly to Drive so future runtimes reuse it.
base_url = (
    "https://storage.googleapis.com/cloud-tpu-checkpoints/detection/"
    "projects/openseg/colab/exported_model/"
)
required_openseg_files = [
    Path("saved_model.pb"),
    Path("variables/variables.index"),
    Path("variables/variables.data-00000-of-00001"),
]
for relative_path in required_openseg_files:
    destination = DRIVE_OPENSEG / relative_path
    if destination.is_file():
        print("Already cached:", relative_path)
        continue
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    print("Downloading:", relative_path)
    urllib.request.urlretrieve(base_url + relative_path.as_posix(), temporary)
    temporary.replace(destination)
print("OpenSeg download finished.")

In [ ]:
# CPU 6. Verify every persistent input before requesting a GPU.
images = list((DRIVE_DATASET / "images_4").glob("*.JPG"))
sparse_files = [DRIVE_DATASET / "sparse/0" / name for name in
                ("cameras.bin", "images.bin", "points3D.bin")]
checkpoint_ply = DRIVE_CHECKPOINT / "point_cloud/iteration_30000/point_cloud.ply"
assert len(images) == 292, f"Expected 292 images, found {len(images)}"
assert all(path.is_file() for path in sparse_files), "COLMAP files are incomplete"
assert checkpoint_ply.is_file(), f"Missing checkpoint: {checkpoint_ply}"
assert all((DRIVE_OPENSEG / path).is_file() for path in required_openseg_files), (
    "OpenSeg model is incomplete"
)
print("CPU preparation complete. All persistent inputs are verified.")

## Stop here and switch to a T4 GPU runtime

After the CPU verification succeeds, select **Runtime → Disconnect and delete runtime**, then **Runtime → Change runtime type → T4 GPU**. Start again from the next cell—not from the top—so downloads are not repeated on paid GPU time. Changing runtimes clears Python variables, so the next cell deliberately remounts Drive and recreates every path.

In [ ]:
# GPU 1. Reinitialize the fresh runtime and verify the T4 before setup.
import shutil
import subprocess
import torch
from pathlib import Path
from google.colab import drive

assert shutil.which("nvidia-smi"), (
    "No NVIDIA GPU was found. Select Runtime > Change runtime type > T4 GPU."
)
drive.mount("/content/drive")
REPO = Path("/content/semantic-gaussians")
CONDA = Path("/content/miniforge3/bin/conda")
DRIVE_ROOT = Path("/content/drive/MyDrive/semantic-gaussians")
DRIVE_DATASET = DRIVE_ROOT / "data/bonsai"
DRIVE_CHECKPOINT = DRIVE_ROOT / "bonsai/checkpoint"
DRIVE_OPENSEG = DRIVE_ROOT / "weights/openseg_exported_clip"
DRIVE_OUTPUT = DRIVE_ROOT / "outputs/bonsai/fused"
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "The notebook Python cannot access CUDA."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

In [ ]:
# GPU 2. Clone the repository and initialize its CUDA submodules.
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--recursive",
         "https://github.com/sharinka0715/semantic-gaussians.git", str(REPO)],
        check=True,
    )
else:
    print("Repository already exists; skipping clone.")
subprocess.run(
    ["git", "submodule", "update", "--init", "--recursive"],
    cwd=REPO, check=True,
)

In [ ]:
# GPU 3. Install Miniforge and create the repository's pinned environment.
import json

if not CONDA.exists():
    installer = Path("/tmp/miniforge.sh")
    subprocess.run(
        ["wget", "-q",
         "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
         "-O", str(installer)], check=True,
    )
    subprocess.run(
        ["bash", str(installer), "-b", "-p", "/content/miniforge3"],
        check=True,
    )
else:
    print("Miniforge already exists; skipping installation.")

envs = json.loads(subprocess.run(
    [str(CONDA), "env", "list", "--json"],
    check=True, text=True, capture_output=True,
).stdout)["envs"]
if not any(Path(path).name == "sega" for path in envs):
    subprocess.run(
        [str(CONDA), "env", "create", "--file", str(REPO / "environment.yml")],
        check=True,
    )
else:
    print("Environment 'sega' already exists; skipping creation.")

In [ ]:
# GPU 4. Install packages and compile the custom CUDA extensions once per server.
marker = Path("/content/.semantic_gaussians_requirements_ready")
if not marker.exists():
    subprocess.run(
        [str(CONDA), "run", "--no-capture-output", "-n", "sega",
         "pip", "install", "-r", "requirements.txt"],
        cwd=REPO, check=True,
    )
    subprocess.run(
        [str(CONDA), "run", "--no-capture-output", "-n", "sega",
         "pip", "install", "huggingface_hub[cli]", "gdown"],
        check=True,
    )
    marker.touch()
else:
    print("Requirements already installed; skipping.")

In [ ]:
# GPU 5. Link cached OpenSeg weights and verify CUDA in the sega environment.
openseg_model = REPO / "weights/openseg_exported_clip"
openseg_model.parent.mkdir(parents=True, exist_ok=True)
if not openseg_model.exists():
    openseg_model.symlink_to(DRIVE_OPENSEG, target_is_directory=True)
assert (openseg_model / "saved_model.pb").is_file(), "OpenSeg model is incomplete"
assert (openseg_model / "variables/variables.index").is_file()
assert (openseg_model / "variables/variables.data-00000-of-00001").is_file()
assert len(list((DRIVE_DATASET / "images_4").glob("*.JPG"))) == 292
assert (DRIVE_CHECKPOINT / "point_cloud/iteration_30000/point_cloud.ply").is_file()

test_code = """
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable inside sega'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
import rgbd_rasterization
import channel_rasterization
from simple_knn._C import distCUDA2
print('All custom CUDA extensions imported successfully.')
"""
subprocess.run(
    [str(CONDA), "run", "--no-capture-output", "-n", "sega",
     "python", "-c", test_code],
    cwd=REPO, check=True,
)
print("GPU environment and all persistent inputs are ready.")

In [ ]:
# GPU 6. Run fusion through sega, not the notebook's base Python.
local_fused = REPO / "bonsai/fused"
local_fused.mkdir(parents=True, exist_ok=True)
command = [
    str(CONDA), "run", "--no-capture-output", "-n", "sega",
    "python", "fusion.py",
    f"scene.scene_path={DRIVE_DATASET}",
    "scene.colmap_images=images_4",
    f"model.model_dir={DRIVE_CHECKPOINT}",
    "model.load_iteration=30000",
    "fusion.img_dim=[779,519]",
    "fusion.num_workers=2",
    "fusion.out_dir=./bonsai/fused",
]
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
# GPU 7. Copy generated features to Drive before deleting the GPU runtime.
fused_files = list((REPO / "bonsai/fused").glob("*.pt"))
assert fused_files, "Fusion did not produce any .pt files"
for source in fused_files:
    shutil.copy2(source, DRIVE_OUTPUT / source.name)
print(f"Copied {len(fused_files)} file(s) to {DRIVE_OUTPUT}")

## Important limitations

OpenSeg creates a 768-dimensional semantic component for every Gaussian. Even with 4× images, the pretrained Bonsai model can approach a free T4's memory limit. If fusion reports CUDA out-of-memory, stop it before trying another configuration. The Bonsai dataset, checkpoint, OpenSeg model, and fused output persist in Drive. The Conda environment, compiled packages, and repository clone remain temporary and must be recreated on a new GPU server. After the output-copy cell succeeds, select **Runtime → Disconnect and delete runtime** immediately.